# 🏦 VIETNAM BANKING NEWS SCRAPER - PIPELINE HOÀN CHỈNH

**Pipeline tự động cào tin tức ngân hàng từ các trang báo uy tín Việt Nam**

## 📰 Nguồn dữ liệu:
- vneconomy.vn
- vtv.vn  
- vnexpress.net
- cafef.vn
- thoibaotaichinhvietnam.vn
- baodautu.vn

## 🎯 Mục tiêu:
- Thu thập tin tức liên quan đến ngân hàng
- Phân tích sentiment cơ bản
- Xuất dữ liệu CSV để training model

---
**Hướng dẫn:** Chỉ cần nhấn "Run All" và chờ kết quả!

In [1]:
# 📦 IMPORT THỦ VIỆN
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import hashlib
import os
from urllib.parse import urljoin, urlparse
import re
from datetime import datetime
import random
import warnings
warnings.filterwarnings('ignore')

print("📦 Đã import thành công tất cả thư viện cần thiết!")
print("🚀 Pipeline sẵn sàng hoạt động!")

📦 Đã import thành công tất cả thư viện cần thiết!
🚀 Pipeline sẵn sàng hoạt động!


In [2]:
# ⚙️ CẤU HÌNH HỆ THỐNG

# Tạo thư mục output
OUTPUT_DIR = "dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 🔧 CẤU HÌNH SCRAPING - ĐIỀU CHỈNH TẠI ĐÂY!
# ============================================
MAX_PAGES_PER_SITE = 20  # 🔥 TĂNG LÊN ĐỂ CÓ NHIỀU DỮ LIỆU HỠN (khuyến nghị: 5-20)
MAX_ARTICLES_PER_PAGE = 25  # 🔥 SỐ BÀI BÁO TỐI ĐA MỖI TRANG (khuyến nghị: 15-30)
DELAY_BETWEEN_REQUESTS = (1, 3)  # Delay ngẫu nhiên giữa các request (giây)
REQUEST_TIMEOUT = 15  # Timeout cho mỗi request

# Headers chống phát hiện bot
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'vi-VN,vi;q=0.9,en;q=0.8',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
}

BANKING_KEYWORDS = [
    # Nhóm 1: Ngân hàng (tên chung + biến thể) – mở rộng với các thuật ngữ cơ bản từ glossary
    'ngân hàng', 'ngan hang', 'nhà băng', 'nha bang',
    'tín dụng', 'tin dung', 'lãi suất', 'lai suat',
    'vay vốn', 'vay von', 'tiền gửi', 'tien gui',
    'thẻ tín dụng', 'the tin dung', 'thanh toán', 'thanh toan',
    'ví điện tử', 'vi dien tu', 'banking', 'bank', 'credit', 'loan', 'fintech',
    'tiền tệ', 'tien te', 'tỷ giá', 'ty gia', 'phí giao dịch', 'phi giao dich',
    'rút tiền', 'rut tien', 'chuyển khoản', 'chuyen khoan', 'thế chấp', 'the chap',
    'đầu tư', 'dau tu', 'lợi nhuận', 'loi nhuan', 'deposit', 'withdraw', 'transfer',
    'atm', 'currency', 'exchange rate', 'transaction fee', 'mortgage', 'investment',
    'profit', 'balance', 'so du', 'remittance', 'su gui tien', 'cash', 'tien mat',
    'service charge', 'phi dich vu', 'repayment', 'khoan tra no', 'account', 'tai khoan',

    # Nhóm 2: Ngân hàng cụ thể (tên, ticker, biến thể) – giữ nguyên như list trước, xác nhận từ nguồn tin tức
    'vietcombank', 'vcb', 'ngoại thương',
    'vietinbank', 'ctg', 'công thương',
    'bidv', 'bid', 'đầu tư và phát triển',
    'agribank', 'agr',
    'techcombank', 'tcb', 'kỹ thương',
    'mbbank', 'mbb', 'quân đội',
    'vpbank', 'vpb', 'việt nam thịnh vượng',
    'acb', 'á châu',
    'tpbank', 'tpb', 'tiên phong',
    'sacombank', 'stb', 'sài gòn thương tín',
    'hdbank', 'hdb', 'phát triển thành phố hồ chí minh',
    'vib', 'ngân hàng quốc tế',
    'shb', 'sài gòn hà nội',
    'ocb', 'phương đông',
    'seabank', 'ssb', 'đông nam á',
    'lienvietpostbank', 'lpb', 'bưu điện liên việt',
    'eximbank', 'eib', 'xuất nhập khẩu',
    'abbank', 'abb', 'an bình',
    'bac a bank', 'bab', 'bắc á',
    'bvbank', 'bvb', 'bản việt',
    'kienlongbank', 'klb', 'kiên long',
    'msb', 'maritime bank', 'hàng hải',
    'nam a bank', 'nab', 'nam á',
    'ncb', 'nvb', 'quốc dân',
    'pgbank', 'pgb', 'xăng dầu petrolimex',
    'saigonbank', 'sgb', 'sài gòn công thương',
    'vietabank', 'vab', 'việt á',
    'vietbank', 'vbb', 'việt nam thương tín',

    # Nhóm 3: Kinh tế - vĩ mô – mở rộng với các chỉ số kinh tế phổ biến
    'kinh tế', 'kinh te', 'phát triển kinh tế', 'phat trien kinh te', 'tăng trưởng', 'tang truong',
    'lạm phát', 'lam phat', 'tỷ giá', 'ty gia', 'ngoại tệ', 'ngoai te', 'usd', 'đô la mỹ', 'do la my',
    'vàng', 'vang', 'giá vàng', 'gia vang', 'trái phiếu', 'trai phieu', 'chứng khoán', 'chung khoan',
    'thị trường chứng khoán', 'thi truong chung khoan', 'bất động sản', 'bat dong san', 'thanh khoản', 'thanh khoan',
    'lãi suất huy động', 'lai suat huy dong', 'lãi suất cho vay', 'lai suat cho vay', 'đầu tư', 'dau tu',
    'thị trường tài chính', 'thi truong tai chinh', 'khủng hoảng tài chính', 'khung hoang tai chinh',
    'cải cách tài chính', 'cai cach tai chinh', 'cổ phiếu', 'co phieu', 'thị trường', 'thi truong',
    'gdp', 'tổng sản phẩm quốc nội', 'tong san pham quoc noi', 'unemployment', 'thất nghiệp', 'that nghiep',
    'recession', 'suy thoái', 'suy thoai', 'deflation', 'giảm phát', 'giam phat', 'economy', 'market economy',
    'nền kinh tế thị trường', 'nen kinh te thi truong', 'import', 'nhập khẩu', 'nhap khau',
    'export', 'xuất khẩu', 'xuat khau', 'trade', 'buôn bán', 'buon ban',

    # Nhóm 4: Chính sách - quản lý – mở rộng với các thuật ngữ quy định
    'ngân hàng nhà nước', 'ngan hang nha nuoc', 'nhnn', 'chính phủ', 'chinh phu', 'bộ tài chính', 'bo tai chinh',
    'chính sách tiền tệ', 'chinh sach tien te', 'nới room tín dụng', 'noi room tin dung',
    'giảm lãi suất', 'giam lai suat', 'tăng lãi suất', 'tang lai suat', 'gói hỗ trợ', 'goi ho tro',
    'cơ cấu nợ', 'co cau no', 'xử lý nợ xấu', 'xu ly no xau', 'quản trị rủi ro', 'quan tri rui ro',
    'basel ii', 'basel iii', 'an toàn vốn', 'an toan von', 'monetary policy', 'fiscal policy',
    'regulation', 'quy định', 'quy dinh', 'compliance', 'tuân thủ', 'tuan thu', 'central bank',
    'state bank of vietnam', 'sbv', 'law on credit institutions', 'luật các tổ chức tín dụng', 'luat cac to chuc tin dung',

    # Nhóm 5: Sản phẩm & Dịch vụ Tài chính – mới, từ glossary
    'tài khoản tiết kiệm', 'tai khoan tiet kiem', 'savings account', 'tài khoản vãng lai', 'tai khoan vang lai',
    'checking account', 'thẻ ghi nợ', 'the ghi no', 'debit card', 'bảo hiểm', 'bao hiem', 'insurance',
    'khoản vay mua nhà', 'khoan vay mua nha', 'home loan', 'quỹ đầu tư', 'quy dau tu', 'investment fund',
    'niên kim', 'nien kim', 'annuity', 'trái phiếu chính phủ', 'trai phieu chinh phu', 'government bond',
    'tài khoản ngân hàng', 'tai khoan ngan hang', 'bank account', 'khoản vay', 'khoan vay', 'loan',
    'lãi suất cố định', 'lai suat co dinh', 'fixed rate', 'lãi suất biến đổi', 'lai suat bien doi', 'variable rate',
    'chuyển tiền', 'chuyen tien', 'wire transfer', 'số dư', 'so du', 'balance', 'phí dịch vụ', 'phi dich vu', 'fee',

    # Nhóm 6: Fintech & Tài chính Kỹ thuật số – mới, phản ánh xu hướng
    'ngân hàng số', 'ngan hang so', 'digital banking', 'ngân hàng di động', 'ngan hang di dong', 'mobile banking',
    'tiền ảo', 'tien ao', 'cryptocurrency', 'blockchain', 'ví điện tử', 'vi dien tu', 'e-wallet',
    'thanh toán trực tuyến', 'thanh toan truc tuyen', 'online payment', 'fintech', 'công nghệ tài chính', 'cong nghe tai chinh',
    'ai', 'trí tuệ nhân tạo', 'tri tue nhan tao', 'big data', 'dữ liệu lớn', 'du lieu lon',
    'p2p lending', 'cho vay ngang hàng', 'cho vay ngang hang', 'robo-advisor', 'cố vấn robot', 'co van robot',

    # Nhóm 7: Rủi ro & Tuân thủ – mới, từ các nguồn rủi ro tài chính
    'nợ xấu', 'no xau', 'bad debt', 'lừa đảo', 'lua dao', 'fraud', 'rủi ro tín dụng', 'rui ro tin dung', 'credit risk',
    'an ninh mạng', 'an ninh mang', 'cybersecurity', 'vi phạm', 'vi pham', 'violation', 'xóa nợ', 'xoa no', 'write-off',
    'khủng hoảng', 'khung hoang', 'crisis', 'usury', 'cho vay nặng lãi', 'cho vay nang lai', 'rủi ro thị trường', 'rui ro thi truong',
    'market risk', 'basel', 'an toàn vốn', 'an toan von', 'capital adequacy', 'quản lý rủi ro', 'quan ly rui ro', 'risk management'
]


# Cấu hình các trang báo
NEWS_SITES = {
    'vneconomy.vn': {
        'base_url': 'https://vneconomy.vn',
        'search_urls': [
            'https://vneconomy.vn/ngan-hang.htm',
            'https://vneconomy.vn/tai-chinh.htm',
            'https://vneconomy.vn/tieu-diem.htm',
            'https://vneconomy.vn/dau-tu.htm',
            'https://vneconomy.vn/kinh-te-so.htm',
            'https://vneconomy.vn/kinh-te-xanh.htm',
            'https://vneconomy.vn/thi-truong.htm',
            'https://vneconomy.vn/nhip-cau-doanh-nghiep.htm',

        ]
    },
    'vtv.vn': {
        'base_url': 'https://vtv.vn',
        'search_urls': [
            'https://vtv.vn/kinh-te.htm',
        ]
    },
    'vnexpress.net': {
        'base_url': 'https://vnexpress.net',
        'search_urls': [
            'https://vnexpress.net/kinh-doanh/ngan-hang',
            'https://vnexpress.net/kinh-doanh',
        ]
    },
    'cafef.vn': {
        'base_url': 'https://cafef.vn',
        'search_urls': [
            'https://cafef.vn/thi-truong-chung-khoan.chn',
            'https://cafef.vn/kinh-te-so.chn',
            'https://cafef.vn/tai-chinh-ngan-hang.chn',
            'https://cafef.vn/thi-truong.chn',
            'https://cafef.vn/doanh-nghiep.chn',

        ]
    },
    'thoibaotaichinhvietnam.vn': {
        'base_url': 'https://thoibaotaichinhvietnam.vn',
        'search_urls': [
            'https://thoibaotaichinhvietnam.vn/kinh-te',
            'https://thoibaotaichinhvietnam.vn/tai-chinh',
            'https://thoibaotaichinhvietnam.vn/thue-hai-quan',
            'https://thoibaotaichinhvietnam.vn/chung-khoan',
            'https://thoibaotaichinhvietnam.vn/ngan-hang',
            'https://thoibaotaichinhvietnam.vn/bao-hiem',
            'https://thoibaotaichinhvietnam.vn/kinh-doanh',
            'https://thoibaotaichinhvietnam.vn/bat-dong-san',
            'https://thoibaotaichinhvietnam.vn/gia-ca',
            'https://thoibaotaichinhvietnam.vn/xa-hoi',
        ]
    },
    'baodautu.vn': {
        'base_url': 'https://baodautu.vn',
        'search_urls': [
            'https://baodautu.vn/special.html',
            'https://baodautu.vn/thoi-su-d1/',
            'https://baodautu.vn/dau-tu-d2/',
            'https://baodautu.vn/batdongsan/',
            'https://baodautu.vn/quoc-te-d54/',
            'https://baodautu.vn/doanh-nghiep-d3/',
            'https://baodautu.vn/doanh-nhan-d4/',
            'https://baodautu.vn/ngan-hang--bao-hiem-d5/',
            'https://baodautu.vn/tai-chinh-chung-khoan-d6/',
        ]
    }
}

print("⚙️ Đã cấu hình hệ thống thành công!")
print(f"📊 Sẽ scrape từ {len(NEWS_SITES)} trang báo uy tín")
print(f"📄 Tối đa {MAX_PAGES_PER_SITE} trang mỗi site = {MAX_PAGES_PER_SITE * len(NEWS_SITES)} trang tổng cộng")
print(f"📰 Tối đa {MAX_ARTICLES_PER_PAGE} bài mỗi trang = {MAX_ARTICLES_PER_PAGE * MAX_PAGES_PER_SITE * len(NEWS_SITES)} bài tối đa")
print(f"📁 Dữ liệu sẽ được lưu trong thư mục: {OUTPUT_DIR}")
print(f"📄 Tên file: web_scrap_news.csv")
print("\\n💡 Muốn thay đổi số lượng? Sửa MAX_PAGES_PER_SITE và MAX_ARTICLES_PER_PAGE ở cell trên!")

⚙️ Đã cấu hình hệ thống thành công!
📊 Sẽ scrape từ 6 trang báo uy tín
📄 Tối đa 20 trang mỗi site = 120 trang tổng cộng
📰 Tối đa 25 bài mỗi trang = 3000 bài tối đa
📁 Dữ liệu sẽ được lưu trong thư mục: dataset
📄 Tên file: web_scrap_news.csv
\n💡 Muốn thay đổi số lượng? Sửa MAX_PAGES_PER_SITE và MAX_ARTICLES_PER_PAGE ở cell trên!


In [3]:
# 🔧 ĐỊNH NGHĨA CÁC HÀM TIỆN ÍCH

def clean_text(text):
    """Làm sạch text"""
    if not text:
        return ""
    # Loại bỏ ký tự đặc biệt và khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text.strip())
    text = re.sub(r'[^\w\s\.\,\?\!\-\:\;\(\)]', ' ', text)
    return text.strip()

def is_banking_related(title, content):
    """Kiểm tra có liên quan đến ngân hàng không"""
    full_text = (title + " " + content).lower()
    return any(keyword.lower() in full_text for keyword in BANKING_KEYWORDS)

def calculate_sentiment_raw(title, content):
    """Tính sentiment đơn giản"""
    positive_words = ['tăng', 'tốt', 'khả quan', 'phát triển', 'lợi nhuận', 
                     'thành công', 'mạnh', 'cao', 'cải thiện', 'tăng trưởng']
    negative_words = ['giảm', 'xấu', 'khó khăn', 'thua lỗ', 'rủi ro', 
                     'suy giảm', 'yếu', 'thấp', 'khủng hoảng', 'thiệt hại']
    
    full_text = (title + " " + content).lower()
    positive_count = sum(1 for word in positive_words if word in full_text)
    negative_count = sum(1 for word in negative_words if word in full_text)
    
    if positive_count > negative_count:
        return 1.0
    elif negative_count > positive_count:
        return -1.0
    else:
        return 0.0

def generate_article_id(title, url, published_date):
    """Tạo ID duy nhất cho bài báo"""
    unique_string = f"{title[:50]}{url}{published_date}"
    return hashlib.md5(unique_string.encode('utf-8')).hexdigest()[:12]

def extract_publish_date(soup, url):
    """Trích xuất ngày đăng từ soup hoặc URL"""
    date_selectors = [
        'meta[property="article:published_time"]',
        'meta[name="pubdate"]',
        'meta[name="date"]',
        '[class*="date"]',
        '[class*="time"]',
        'time'
    ]
    
    for selector in date_selectors:
        date_elem = soup.select_one(selector)
        if date_elem:
            date_text = date_elem.get('content') or date_elem.get_text()
            if date_text:
                try:
                    for fmt in ['%Y-%m-%d', '%d/%m/%Y', '%d-%m-%Y']:
                        try:
                            parsed_date = datetime.strptime(date_text[:10], fmt)
                            return parsed_date.strftime('%Y-%m-%d')
                        except:
                            continue
                except:
                    pass
    
    # Nếu không tìm thấy, dùng ngày hiện tại
    return datetime.now().strftime('%Y-%m-%d')

print("🔧 Đã định nghĩa tất cả hàm tiện ích!")

🔧 Đã định nghĩa tất cả hàm tiện ích!


In [4]:
# 🕷️ ĐỊNH NGHĨA CLASS SCRAPER CHÍNH

class VietnamBankingNewsScraper:
    """Class scraper tin tức ngân hàng Việt Nam hoàn chỉnh"""
    
    def __init__(self, output_dir=OUTPUT_DIR):
        self.output_dir = output_dir
        self.session = None  # Lazy initialization - tạo khi cần
        self.articles_scraped = []
    
    def _ensure_session(self):
        """Tạo session khi cần thiết (lazy loading)"""
        if self.session is None:
            self.session = requests.Session()
            self.session.headers.update(HEADERS)
        
    def find_article_links(self, url):
        """Tìm các link bài báo từ trang danh sách"""
        try:
            self._ensure_session()  # Chỉ tạo session khi cần
            time.sleep(random.uniform(*DELAY_BETWEEN_REQUESTS))
            
            response = self.session.get(url, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')
            
            links = set()
            
            for a in soup.find_all('a', href=True):
                href = a['href']
                
                # Chuyển thành URL đầy đủ
                if href.startswith('/'):
                    parsed = urlparse(url)
                    full_url = f"{parsed.scheme}://{parsed.netloc}{href}"
                elif href.startswith('http'):
                    full_url = href
                else:
                    continue
                
                # Lọc các link có vẻ là bài báo
                if any(ext in href for ext in ['.htm', '.html', '.aspx']) or '/news/' in href or '/kinh-te/' in href:
                    title_text = a.get_text().strip()
                    if title_text and len(title_text) > 10:
                        if any(keyword.lower() in title_text.lower() for keyword in BANKING_KEYWORDS[:10]):  # Chỉ check 10 keyword đầu
                            links.add(full_url)
            
            return list(links)[:MAX_ARTICLES_PER_PAGE]
            
        except Exception as e:
            print(f"    ❌ Lỗi tìm link từ {url}: {e}")
            return []
    
    def scrape_article_detail(self, url):
        """Scrape chi tiết một bài báo"""
        try:
            self._ensure_session()  # Chỉ tạo session khi cần
            time.sleep(random.uniform(0.5, 1.5))
            
            response = self.session.get(url, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Tìm tiêu đề
            title = ""
            title_selectors = [
                'h1', 'h2', 
                '[class*="title"]', '[class*="headline"]',
                '[id*="title"]', '.entry-title'
            ]
            for selector in title_selectors:
                title_elem = soup.select_one(selector)
                if title_elem:
                    title = clean_text(title_elem.get_text())
                    if title and len(title) > 5:
                        break
            
            if not title:
                return None
            
            # Tìm nội dung
            content = ""
            content_selectors = [
                '[class*="content"]', '[class*="body"]', 
                '[class*="article"]', '[id*="content"]',
                '.entry-content', '.post-content',
                'p'
            ]
            
            for selector in content_selectors:
                content_elems = soup.select(selector)
                if content_elems:
                    content_texts = []
                    for elem in content_elems:
                        text = clean_text(elem.get_text())
                        if text and len(text) > 20:
                            content_texts.append(text)
                    
                    if content_texts:
                        content = " ".join(content_texts)[:2000]
                        break
            
            # Kiểm tra có liên quan đến ngân hàng không
            if not is_banking_related(title, content):
                return None
            
            # Trích xuất ngày đăng
            published_date = extract_publish_date(soup, url)
            
            # Tạo article data
            article_data = {
                'article_id': generate_article_id(title, url, published_date),
                'published_at': published_date,
                'source': urlparse(url).netloc,
                'title': title,
                'content': content,
                'sentiment_raw': calculate_sentiment_raw(title, content),
                'url': url
            }
            
            return article_data
            
        except Exception as e:
            return None
    
    def scrape_website(self, site_name, site_config, max_pages=MAX_PAGES_PER_SITE):
        """Scrape một website"""
        print(f"\n🌐 Đang scrape {site_name}...")
        articles = []
        
        for i, search_url in enumerate(site_config['search_urls'][:max_pages]):
            print(f"  📄 Trang {i+1}/{min(len(site_config['search_urls']), max_pages)}: {search_url}")
            
            # Tìm các link bài báo
            article_links = self.find_article_links(search_url)
            print(f"    🔗 Tìm thấy {len(article_links)} link bài báo")
            
            # Scrape từng bài báo
            scraped_count = 0
            for link in article_links:
                article = self.scrape_article_detail(link)
                if article:
                    articles.append(article)
                    scraped_count += 1
                    print(f"    ✅ [{scraped_count:2d}] {article['title'][:60]}...")
        
        return articles
    
    def run_full_scraping(self):
        """Chạy scraping toàn bộ"""
        print("🚀 BẮT ĐẦU SCRAPING TOÀN BỘ CÁC TRANG BÁO UY TÍN VIỆT NAM")
        print("=" * 70)
        
        start_time = time.time()
        all_articles = []
        
        for site_name, site_config in NEWS_SITES.items():
            try:
                articles = self.scrape_website(site_name, site_config)
                all_articles.extend(articles)
                print(f"📊 {site_name}: {len(articles)} bài báo")
                
            except Exception as e:
                print(f"❌ Lỗi scraping {site_name}: {e}")
        
        end_time = time.time()
        duration = end_time - start_time
        
        print(f"\n🎉 HOÀN THÀNH SCRAPING!")
        print(f"⏱️ Thời gian: {duration:.1f} giây")
        print(f"📊 Tổng cộng: {len(all_articles)} bài báo")
        
        # Lưu dữ liệu
        if all_articles:
            self.save_to_csv(all_articles)
            self.print_statistics(all_articles)
        else:
            print("❌ Không thu thập được bài báo nào!")
        
        return all_articles
    
    def save_to_csv(self, articles):
        """Lưu articles vào CSV với logging chi tiết"""
        if not articles:
            print("❌ Không có bài báo nào để lưu")
            return
        
        print(f"\n💾 BẮT ĐẦU LƯU DỮ LIỆU...")
        print(f"📊 Tổng số bài báo thu thập: {len(articles)}")
        
        # Tạo DataFrame
        df = pd.DataFrame(articles)
        print(f"✅ Đã tạo DataFrame với {len(df)} dòng")
        
        # Kiểm tra article_id trùng lặp TRƯỚC khi loại bỏ
        duplicate_ids = df[df.duplicated(subset=['article_id'], keep=False)]
        if len(duplicate_ids) > 0:
            print(f"⚠️ Phát hiện {len(duplicate_ids)} bài báo có article_id trùng lặp:")
            for idx, row in duplicate_ids.iterrows():
                print(f"  🔄 ID: {row['article_id']} - {row['title'][:50]}...")
        
        # Sắp xếp theo ngày
        df = df.sort_values('published_at', ascending=False)
        print(f"✅ Đã sắp xếp theo ngày đăng")
        
        # Loại bỏ trùng lặp và đếm số lượng bị loại bỏ
        original_count = len(df)
        df = df.drop_duplicates(subset=['article_id'], keep='first')
        duplicates_removed = original_count - len(df)
        
        if duplicates_removed > 0:
            print(f"🗑️ Đã loại bỏ {duplicates_removed} bài báo trùng lặp")
        
        print(f"📊 Số bài báo cuối cùng: {len(df)}")
        
        # Lưu file
        csv_path = os.path.join(self.output_dir, "web_scrap_news.csv")
        try:
            df.to_csv(csv_path, index=False, encoding='utf-8-sig')
            actual_file_size = os.path.getsize(csv_path) / 1024
            print(f"✅ Đã lưu thành công {len(df)} bài báo vào {csv_path}")
            print(f"💾 Kích thước file: {actual_file_size:.1f} KB")
            print(f" Cấu trúc dữ liệu: {list(df.columns)}")
            
            # Kiểm tra lại file vừa lưu
            df_check = pd.read_csv(csv_path)
            if len(df_check) != len(df):
                print(f"❌ CẢNH BÁO: File lưu có {len(df_check)} dòng, khác với DataFrame gốc {len(df)} dòng!")
            else:
                print(f"✅ Xác nhận: File đã lưu chính xác {len(df_check)} bài báo")
                
        except Exception as e:
            print(f"❌ Lỗi khi lưu file CSV: {e}")
    
    def print_statistics(self, articles):
        """In thống kê"""
        if not articles:
            return
        
        df = pd.DataFrame(articles)
        
        print(f"\n📊 THỐNG KÊ CHI TIẾT")
        print("=" * 50)
        print(f"📰 Tổng số bài báo: {len(df)}")
        
        # Thống kê theo nguồn
        source_stats = df['source'].value_counts()
        print(f"\n📈 Phân bố theo nguồn:")
        for source, count in source_stats.items():
            percentage = (count/len(df))*100
            print(f"  {source:30s}: {count:3d} bài ({percentage:5.1f}%)")
        
        # Thống kê sentiment
        sentiment_stats = df['sentiment_raw'].value_counts()
        print(f"\n😊 Phân tích sentiment:")
        sentiment_labels = {1.0: 'Tích cực', 0.0: 'Trung tính', -1.0: 'Tiêu cực'}
        for sentiment, count in sentiment_stats.items():
            label = sentiment_labels.get(sentiment, f'Khác ({sentiment})')
            percentage = (count/len(df))*100
            print(f"  {label:30s}: {count:3d} bài ({percentage:5.1f}%)")

# ✅ ĐỊNH NGHĨA CLASS HOÀN TẤT - KHÔNG THỰC THI GÌ CẢ
print("🕷️ Đã định nghĩa class VietnamBankingNewsScraper với lazy loading!")
print("⚡ Class chỉ định nghĩa, không tạo kết nối mạng ngay!")

🕷️ Đã định nghĩa class VietnamBankingNewsScraper với lazy loading!
⚡ Class chỉ định nghĩa, không tạo kết nối mạng ngay!


In [5]:
# 🚀 CHẠY PIPELINE HOÀN CHỈNH

print("🎯 KHỞI ĐỘNG PIPELINE TỰ ĐỘNG CÀO TIN TỨC NGÂN HÀNG VIỆT NAM")
print("=" * 70)

# Khởi tạo scraper
scraper = VietnamBankingNewsScraper()
print("✅ Đã khởi tạo scraper thành công!")

# Chạy scraping tự động
articles = scraper.run_full_scraping()

print("\n" + "=" * 70)
print("🎊 PIPELINE HOÀN THÀNH!")
print(f"📁 Dữ liệu đã được lưu trong thư mục: {OUTPUT_DIR}")
print(f"📄 File CSV: web_scrap_news.csv")
print("\n🎯 Dữ liệu sẵn sàng để training sentiment analysis model!")

🎯 KHỞI ĐỘNG PIPELINE TỰ ĐỘNG CÀO TIN TỨC NGÂN HÀNG VIỆT NAM
✅ Đã khởi tạo scraper thành công!
🚀 BẮT ĐẦU SCRAPING TOÀN BỘ CÁC TRANG BÁO UY TÍN VIỆT NAM

🌐 Đang scrape vneconomy.vn...
  📄 Trang 1/8: https://vneconomy.vn/ngan-hang.htm
    🔗 Tìm thấy 1 link bài báo
    ✅ [ 1] OCB trở thành thành viên chính thức của liên minh Ngân hàng ...
  📄 Trang 2/8: https://vneconomy.vn/tai-chinh.htm
    🔗 Tìm thấy 1 link bài báo
    ✅ [ 1] OCB trở thành thành viên chính thức của liên minh Ngân hàng ...
  📄 Trang 3/8: https://vneconomy.vn/tieu-diem.htm
    🔗 Tìm thấy 1 link bài báo
    ✅ [ 1] Ngân hàng rục rịch nhập khẩu vàng nguyên liệu và sản xuất và...
  📄 Trang 4/8: https://vneconomy.vn/dau-tu.htm
    🔗 Tìm thấy 0 link bài báo
  📄 Trang 5/8: https://vneconomy.vn/kinh-te-so.htm
    🔗 Tìm thấy 0 link bài báo
  📄 Trang 6/8: https://vneconomy.vn/kinh-te-xanh.htm
    🔗 Tìm thấy 0 link bài báo
  📄 Trang 7/8: https://vneconomy.vn/thi-truong.htm
    🔗 Tìm thấy 0 link bài báo
  📄 Trang 8/8: https://vneconom

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


    🔗 Tìm thấy 0 link bài báo
  📄 Trang 2/9: https://baodautu.vn/thoi-su-d1/


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


    🔗 Tìm thấy 0 link bài báo
  📄 Trang 3/9: https://baodautu.vn/dau-tu-d2/


...

    🔗 Tìm thấy 0 link bài báo
  📄 Trang 8/9: https://baodautu.vn/ngan-hang--bao-hiem-d5/


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


    🔗 Tìm thấy 0 link bài báo
  📄 Trang 9/9: https://baodautu.vn/tai-chinh-chung-khoan-d6/


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


    🔗 Tìm thấy 0 link bài báo
📊 baodautu.vn: 0 bài báo

🎉 HOÀN THÀNH SCRAPING!
⏱️ Thời gian: 175.3 giây
📊 Tổng cộng: 60 bài báo

💾 BẮT ĐẦU LƯU DỮ LIỆU...
📊 Tổng số bài báo thu thập: 60
✅ Đã tạo DataFrame với 60 dòng
⚠️ Phát hiện 54 bài báo có article_id trùng lặp:
  🔄 ID: 488c85e6bba0 - OCB trở thành thành viên chính thức của liên minh ...
  🔄 ID: 488c85e6bba0 - OCB trở thành thành viên chính thức của liên minh ...
  🔄 ID: 27de792d05e9 - VPBank sắp tham gia thị trường tài sản mã hóa...
  🔄 ID: 7ecaea035cdc - Châu Âu tiếp tục giữ nguyên lãi suất...
  🔄 ID: cf876ffe464c - Ngân hàng Nhà nước yêu cầu khắc phục triệt để các ...
  🔄 ID: 44a3de560c52 - Ngân hàng Nhà nước đang nghiên cứu lập sàn giao dị...
  🔄 ID: 674cfe22eacd - Chính phủ yêu cầu hoàn thiện phương án cơ cấu lại ...
  🔄 ID: 56ab5f121151 - Yêu cầu báo cáo giải pháp quản lý thị trường vàng,...
  🔄 ID: b1b01f065fae - Bà Nguyễn Thị Hồng là một trong ba Thống đốc được ...
  🔄 ID: 504d225eb107 - Cổ phiếu trụ giúp chứng khoán đảo chiề

In [6]:
# 📋 KIỂM TRA KẾT QUẢ CUỐI CÙNG

csv_file = os.path.join(OUTPUT_DIR, "web_scrap_news.csv")

if os.path.exists(csv_file):
    print("📄 KIỂM TRA FILE DỮ LIỆU CUỐI CÙNG")
    print("=" * 50)
    
    # Đọc file CSV
    df = pd.read_csv(csv_file)
    
    print(f"✅ File tồn tại: {csv_file}")
    print(f"📊 Số bài báo: {len(df)}")
    print(f"📋 Các cột: {list(df.columns)}")
    print(f"💾 Kích thước file: {os.path.getsize(csv_file) / 1024:.1f} KB")
    
    # Hiển thị mẫu dữ liệu
    print(f"\n📋 SAMPLE 3 BÀI BÁO:")
    print("-" * 80)
    for i, row in df.head(3).iterrows():
        print(f"🆔 ID: {row['article_id']}")
        print(f"📅 Ngày: {row['published_at']}")
        print(f"📰 Nguồn: {row['source']}")
        print(f"📝 Tiêu đề: {row['title'][:70]}...")
        print(f"😊 Sentiment: {row['sentiment_raw']}")
        print(f"🔗 URL: {row['url'][:50]}...")
        print("-" * 40)
    
    print(f"\n✅ HOÀN TẤT! Dữ liệu sẵn sàng cho sentiment analysis!")
    
else:
    print(f"❌ Không tìm thấy file {csv_file}")
    print("🔄 Hãy chạy lại pipeline để tạo dữ liệu")

📄 KIỂM TRA FILE DỮ LIỆU CUỐI CÙNG
✅ File tồn tại: dataset\web_scrap_news.csv
📊 Số bài báo: 21
📋 Các cột: ['article_id', 'published_at', 'source', 'title', 'content', 'sentiment_raw', 'url']
💾 Kích thước file: 39.1 KB

📋 SAMPLE 3 BÀI BÁO:
--------------------------------------------------------------------------------
🆔 ID: 488c85e6bba0
📅 Ngày: 2025-09-12
📰 Nguồn: vneconomy.vn
📝 Tiêu đề: OCB trở thành thành viên chính thức của liên minh Ngân hàng thương mại...
😊 Sentiment: 1.0
🔗 URL: https://vneconomy.vn/ocb-tro-thanh-thanh-vien-chin...
----------------------------------------
🆔 ID: ad64e27937aa
📅 Ngày: 2025-09-12
📰 Nguồn: thoibaotaichinhvietnam.vn
📝 Tiêu đề: Trung tâm Thông tin tín dụng quốc gia CIC bị hacker tấn công...
😊 Sentiment: 0.0
🔗 URL: https://thoibaotaichinhvietnam.vn/trung-tam-thong-...
----------------------------------------
🆔 ID: 60e6eed6f289
📅 Ngày: 2025-09-12
📰 Nguồn: thoibaotaichinhvietnam.vn
📝 Tiêu đề: Các ngân hàng, trung gian thanh toán siết kiểm soát giao dịch tiền